In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
from vbn_utils import formatFigure
import vbn_utils as vbn
import ccf_utils
%matplotlib inline
from functools import partial
import decoding_utils as du
from analysis_utils import exponential_convolve
from scipy.stats import binned_statistic_2d
from scipy.optimize import curve_fit

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
stims = pd.read_csv(stim_table_file)
units = pd.read_csv(unit_table_file)
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/ccf_structure_tree_2017.csv")
units_rf = pd.read_csv(unit_table_with_rf_stats)
rf_cols = [c for c in units_rf.columns if 'rf' in c]
units = units.merge(units_rf[rf_cols + ['unit_id']], on='unit_id', how='left')
rf_array_dir = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/rfs/arrays"
all_rf_files = os.listdir(rf_array_dir)
sessions = pd.read_csv(sessions_table_file)

## put some basic numbers on dataset

In [ ]:
quality = du.apply_unit_quality_filter(units, no_abnorm=True)
quality_units = units[quality]

In [ ]:
quality = du.apply_unit_quality_filter(units)
inregion = du.getUnitsInRegion(units, 'VISall')
ctx = units.loc[quality&inregion]

df = {'cell type': ['RS', 'FS', 'SST', 'VIP'], 
        'Criteria': ['Not optotagged; spike duration > 0.4 ms', 
                                                            'Not optotagged; spike duration < 0.4 ms',
                                                            'Optotagged in SST-cre x Ai32 mouse',
                                                            'Optotagged in VIP-cre x Ai32 mouse'], 
        'Count': [ctx['RS'].sum(), ctx['FS'].sum(), ctx['SST'].sum(), ctx['VIP'].sum()]}

df = pd.DataFrame(df)
display(df)

In [ ]:
mice = sessions.groupby('mouse_id').head(1).reset_index(drop=True)
mice.value_counts(['genotype', 'sex'])

## population RFs across sessions

In [ ]:
from notebook_utils import gaussian_2d

In [ ]:
no_abnorm_sessions = sessions[sessions['abnormal_activity'].isnull() & sessions['abnormal_histology'].isnull()]['ecephys_session_id'].values
quality = du.apply_unit_quality_filter(units, no_abnorm=True)

rf_centers = {area: [] for area in ['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl', 'LGd','LP']}
for isess, session_id in enumerate(no_abnorm_sessions):
    print(isess)
    session_filter = units['ecephys_session_id'] == session_id
    for area in ['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl', 'LGd','LP']:
        in_area = du.getUnitsInRegion(units, area)
        sig_rf = units['p_value_rf'] < 0.05
        on_screen = units['on_screen_rf']

        selected = units[in_area & sig_rf & on_screen & quality & session_filter]

        selected_rf_files= [r for r in all_rf_files if int(r.split('.npy')[0]) in selected['unit_id'].values]
        if len(selected_rf_files) == 0:
            continue
        selected_rfs = []
        for rf_file in selected_rf_files:
            rf = np.load(os.path.join(rf_array_dir, rf_file))
            selected_rfs.append(rf)

        pop_rf = np.mean(selected_rfs, axis=0)
        maxloc = np.unravel_index(np.argmax(pop_rf), pop_rf.shape)

        initial_guess = (pop_rf.max(), maxloc[1], maxloc[0], 2, 2, 0, pop_rf.min())

        try:
            fit_params, pcov = curve_fit(gaussian_2d, np.meshgrid(range(9), range(9)), pop_rf.ravel(), p0=initial_guess, maxfev=10000)

            rf_centers[area].append((fit_params[1]*10+10, 50-fit_params[2]*10)) #transform to degrees
        except:
            print(f'Failed to fit {area} in session {session_id}')
            continue


In [ ]:
from matplotlib.ticker import MaxNLocator
plt.rcParams['font.size'] = 14

figs = []
axes = []
for area in ['VISp', 'VISl', 'VISal', 'VISpm', 'VISam', 'VISrl', 'LGd','LP']:
    rf_centers[area] = np.array(rf_centers[area])
    fig, ax = plt.subplots(1,2)
    fig.set_size_inches(8, 4)
    binnedarray, xedges, yedges, bins = binned_statistic_2d(rf_centers[area][:,0], rf_centers[area][:,1], 
                                                            rf_centers[area][:,0], statistic='count',
                                                            bins=[range(5, 100, 10), range(-35, 60, 10)])
    binnedarray = binnedarray/rf_centers[area].shape[0]
    clims = [0, 0.25] if area == 'VISp' else [0, 0.15]
    im = ax[1].imshow(binnedarray.T, extent=[5, 95, -35, 55],cmap='Greys', origin='lower')

    print(f'{area}: {rf_centers[area].shape[0]} sessions')
    ax[0].plot(rf_centers[area][:,0], rf_centers[area][:,1], 'ko', alpha=0.5)
    ax[0].set_xlim(0, 100)
    ax[0].set_ylim(-40, 60)

    ax[1].tick_params(left=False, right=False, bottom=False, top=False, labelleft=False, labelbottom=False)


    colorbar = fig.colorbar(im, orientation='horizontal', pad=0.1, aspect=25)  # Horizontal colorbar
    colorbar.ax.set_position([ax[1].get_position().x0, ax[1].get_position().y1, ax[1].get_position().width, 0.05])  # Adjust position
    colorbar.ax.xaxis.set_ticks_position('top')  # Move ticks to the top
    colorbar.ax.xaxis.set_label_position('top')  # Move label to the top
    colorbar.locator = MaxNLocator(nbins=3)  # Automatically determine 3 ticks
    colorbar.update_ticks()  # Update the ticks after setting the locator

    axes.append(ax)
    figs.append(fig)

for ia, area in enumerate(['VISp','VISl', 'VISal', 'VISpm', 'VISam', 'VISrl', 'LGd','LP']):
    figs[ia].savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/Figure 1", f'{area}_session_rfs_test.pdf'))


## area unit counts

In [ ]:
quality = du.apply_unit_quality_filter(units, no_abnorm=True)
quality_units = units[quality]

quality_units = quality_units[quality_units['brain_division'] != 'not in list']
quality_units.loc[quality_units['structure_acronym'].isin(['SCig', 'SCiw']).astype(bool), 'structure_acronym'] = 'SCm'
quality_units.loc[quality_units['structure_acronym'].isin(['MGv', 'MGd', 'MGm']).astype(bool), 'structure_acronym'] = 'MG'

areas_meet_threshold = quality_units['structure_acronym'].value_counts() > 1000
areas_meet_threshold = areas_meet_threshold[areas_meet_threshold].index

quality_units = quality_units[quality_units['structure_acronym'].isin(areas_meet_threshold)]
quality_units['structure_acronym'].value_counts()
brain_divisions = ['Thalamus', 'Isocortex', 'Hippocampal formation', 'Midbrain', 'Hypothalamus']

fig, ax = plt.subplots()
area_count = 0
area_labels = []
for bd in brain_divisions:
    bd_units = quality_units[quality_units['brain_division'] == bd]
    bd_areas = np.sort(bd_units['structure_acronym'].unique())
    for area in bd_areas:
        area_units = bd_units[bd_units['structure_acronym'] == area]
        ax.bar(area_count, len(area_units), color=ccf_utils.get_area_color(area, structure_tree))
        area_count += 1
        area_labels.append(area)

ax.set_xticks(np.arange(len(area_labels)))
ax.set_xticklabels(area_labels, rotation=90)
formatFigure(fig, ax)
ax.set_ylabel('Number of units')
fig.savefig("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/Figure 1/units_per_area_test.pdf")

## behavior characterization

In [ ]:
good_sessions = sessions[sessions['abnormal_activity'].isnull() & sessions['abnormal_histology'].isnull()]
good_stims = stims[stims['session_id'].isin(good_sessions['ecephys_session_id'].values)]
good_stims = good_stims.merge(good_sessions[['ecephys_session_id', 'mouse_id']], left_on='session_id', right_on='ecephys_session_id')

mouse_fas = {}
mouse_hits = {}
mouse_dprimes = {}
for mouse in good_stims['mouse_id'].unique():
    mouse_stims = good_stims[good_stims['mouse_id'] == mouse]
    mouse_trials = mouse_stims.groupby('session_id').apply(lambda x: x.groupby('behavior_trial_id').first())
    if (mouse_trials['engaged']&mouse_trials['catch']).sum() > 5:
        mouse_fas[mouse] = (mouse_trials['engaged']&mouse_trials['false_alarm']).sum()/(mouse_trials['engaged']&mouse_trials['catch']).sum()
        mouse_hits[mouse] = (mouse_trials['engaged']&mouse_trials['hit']).sum()/(mouse_trials['engaged']&mouse_trials['go']).sum()
        mouse_dprimes[mouse] = vbn.calcDprime((mouse_trials['engaged']&mouse_trials['hit']).sum(),
                                              (mouse_trials['engaged']&mouse_trials['miss']).sum(),
                                              (mouse_trials['engaged']&mouse_trials['false_alarm']).sum(),
                                              (mouse_trials['engaged']&mouse_trials['correct_reject']).sum())
    else:
        print(f'Mouse {mouse} does not have enough trials to calculate d-prime: {(mouse_trials["engaged"]&mouse_trials["catch"]).sum()} catch trials')

In [ ]:
fig, axes = plt.subplots(1,2)
fig.set_size_inches(5, 4)
hits = []
fas = []
jitters = []
for mouse in mouse_fas.keys():
    jitter = np.random.rand()*0.2 - 0.1
    jitters.append(jitter)
    
    mhit = mouse_hits[mouse]
    mfa = mouse_fas[mouse]

    hits.append(mhit)
    fas.append(mfa)
    axes[0].plot([0+jitter, 1+jitter], [mhit, mfa], '-', color='k',alpha=0.5)

for im, mouse in enumerate(mouse_fas.keys()):
    jitter = jitters[im]
    
    mhit = mouse_hits[mouse]
    mfa = mouse_fas[mouse]

    axes[0].plot([0+jitter, 1+jitter], [mhit, mfa], 'wo', mec='k',alpha=1, ms=10)

axes[0].set_xlim(-0.3,1.3)
axes[0].errorbar([0, 1], [np.mean(hits), np.mean(fas)], yerr=[np.std(hits)/(len(hits)**0.5), np.std(fas)/(len(fas)**0.5)], color='r', fmt='_', ms=10)
axes[0].set_title(f'{scipy.stats.wilcoxon(hits, fas)} \n n = {len(mouse_fas.keys())}')
axes[0].set_xticks([0,1])
axes[0].set_xticklabels(['Hits', 'False Alarms'])
formatFigure(fig, axes[0], xLabel='Response Type', yLabel='Response Rate')


for im, mouse in enumerate(mouse_fas.keys()):
    jitter = jitters[im]
    axes[1].plot([0+jitter], [mouse_dprimes[mouse]], 'wo', mec='k',alpha=1, ms=10)

dprimes = [d for m,d in mouse_dprimes.items()]

axes[1].set_xlim(-0.5,0.5)
axes[1].set_xticks([0])
axes[1].set_ylim(0, 3)
axes[1].errorbar([0], [np.mean(dprimes)], yerr=[np.std(dprimes)/(len(dprimes)**0.5)], color='r', fmt='_', ms=10)

formatFigure(fig, axes[1], yLabel='dprime')

plt.tight_layout()
fig.savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/Figure 1", 'hit_fa_rates.pdf'))

np.mean(dprimes), np.std(dprimes)

In [ ]:
behavior_sessions = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/behavior_sessions.csv")

mice = good_sessions['mouse_id'].unique()

mouse_passing_sessions = {}
for mouse in mice:
    mouse_sessions = behavior_sessions[behavior_sessions['mouse_id'] == mouse]

    static_grating_sessions = np.sum(mouse_sessions['session_type'].str.contains('TRAINING_1'))
    flashed_grating_sessions = np.sum(mouse_sessions['session_type'].str.contains('TRAINING_2'))
    images_sessions =np.sum((mouse_sessions['session_type'].str.contains('TRAINING_3')) | \
                         (mouse_sessions['session_type'].str.contains('TRAINING_4')) | \
                        ((mouse_sessions['session_type'].str.contains('TRAINING_5')) & (mouse_sessions['session_type'].str.contains('epilogue'))))
    
    mouse_passing_sessions[mouse] = {'static_grating': static_grating_sessions, 'flashed_grating': flashed_grating_sessions, 'images': images_sessions}


In [ ]:
fig, ax = plt.subplots()
jitters = []

for mouse in mouse_passing_sessions.keys():
    jitter = np.random.rand()*0.3 - 0.15
    jitters.append(jitter)
    ax.plot(np.arange(3)+jitter, [mouse_passing_sessions[mouse]['static_grating'], 
                        mouse_passing_sessions[mouse]['flashed_grating'], 
                        mouse_passing_sessions[mouse]['images']], '-', color='k',alpha=0.5)

for im, mouse in enumerate(mouse_passing_sessions.keys()):
    jitter = jitters[im]
    ax.plot(np.arange(3)+jitter, [mouse_passing_sessions[mouse]['static_grating'], 
                        mouse_passing_sessions[mouse]['flashed_grating'], 
                        mouse_passing_sessions[mouse]['images']], 'wo',mec='k',alpha=1, ms=10)


ax.errorbar([0, 1, 2], [np.mean([mouse_passing_sessions[mouse][stage] for mouse in mouse_passing_sessions.keys()])
                        for stage in ['static_grating', 'flashed_grating', 'images']], 
                    yerr=[np.std([mouse_passing_sessions[mouse][stage] for mouse in mouse_passing_sessions.keys()])/len(mouse_passing_sessions.keys())**0.5
                        for stage in ['static_grating', 'flashed_grating', 'images']], color='r', fmt='_', ms=10)


formatFigure(fig, ax)
fig.savefig("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/Figure 1/training_times_test.pdf")

total_sessions_to_pass = []
for mouse, mdata in mouse_passing_sessions.items():
    total_sessions_to_pass.append(mdata['static_grating'] + mdata['flashed_grating'] + mdata['images'])
np.mean(total_sessions_to_pass), np.std(total_sessions_to_pass)